# Aplicações com YOLO

Nos materiais anteriores, treinamos e usamos detectores de objetos. Um detector responde **onde estão os objetos** em uma imagem, mas quase nunca é isso que um sistema real precisa saber. Um estacionamento quer saber quantas vagas estão livres; uma prefeitura, quantos carros passam por uma avenida por hora. Entre a saída do detector e a resposta que interessa, há uma camada de **lógica de aplicação**, e é ela que vamos escrever hoje.

Vamos montar duas aplicações, ambas com o YOLO26 pré-treinado no COCO, sem treinar nada. O YOLO26 precisa da Ultralytics 8.4 ou mais nova (`pip install -U ultralytics`); o rastreador instala sozinho o pacote `lap` na primeira vez que é usado.

1. **Monitor de estacionamento**: em uma foto, quais vagas estão ocupadas;
2. **Contador de tráfego**: em um vídeo, quantos veículos de cada tipo passam por uma linha, em cada sentido.

O notebook já traz a detecção, o rastreamento, o desenho, a leitura dos vídeos e as funções geométricas sobre caixas. Em cada aplicação, uma função marcada com `TODO` fica por sua conta: é a regra que transforma detecções em resposta. Todas trabalham só com **bounding boxes** no formato `[x0, y0, x1, y1]`, em pixels, com $(x_0, y_0)$ o canto superior esquerdo e $(x_1, y_1)$ o inferior direito. Cada uma tem uma célula de teste logo depois, com caixas de brinquedo, para conferir a implementação antes de usar nos dados reais.

In [ ]:
import os
import time
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as IPImage
from IPython.display import display
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Ferramentas

### Dados

Usamos dois vídeos de exemplo distribuídos pela Roboflow e três fotos do **PKLot**, um dataset de estacionamentos da UFPR e da PUC-PR. A `download` baixa cada arquivo uma vez para a pasta `data`.

In [ ]:
DATA_DIR = "data"
VIDEO_URL = "https://media.roboflow.com/supervision/video-examples/"
PKLOT_URL = "https://huggingface.co/datasets/Voxel51/PKLot/resolve/main/data/"


def download(url, name):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        os.makedirs(DATA_DIR, exist_ok=True)
        print(f"Baixando {name}...")
        urllib.request.urlretrieve(url, path)
    return path


traffic_video = download(VIDEO_URL + "vehicles.mp4", "vehicles.mp4")

### Detecção e rastreamento

A `detect` roda o YOLO em uma imagem e devolve uma lista de detecções, cada uma um dicionário com o nome da classe (`label`), a confiança (`conf`) e a caixa (`box`). O argumento `classes` filtra as classes do COCO que interessam, pelo nome.

Em vídeo, detectar não basta: precisamos saber que o carro do frame 10 é o mesmo do frame 11. Esse é o problema do **rastreamento** (*tracking*). O rastreador recebe as detecções de cada frame e as associa às do frame anterior, pela posição das caixas e pela aparência, e dá a cada objeto um **identificador** (`id`) que se mantém enquanto ele aparece no vídeo. A Ultralytics traz o rastreador BoT-SORT embutido no método `track`. O rastreador guarda o estado entre um frame e outro, então usamos um modelo separado para ele, criado de novo a cada vídeo.

In [ ]:
detector = YOLO("yolo26n.pt")
class_ids = {name: index for index, name in detector.names.items()}


def to_detections(result):
    detections = []
    for k in range(len(result.boxes)):
        box = result.boxes.xyxy[k].tolist()   # [x0, y0, x1, y1]
        detection = {
            "label": result.names[int(result.boxes.cls[k])],
            "conf": float(result.boxes.conf[k]),
            "box": box,
        }
        if result.boxes.id is not None:
            detection["id"] = int(result.boxes.id[k])
        detections.append(detection)
    return detections


def detect(image, classes, imgsz=640, conf=0.25):
    result = detector.predict(image, classes=[class_ids[name] for name in classes], imgsz=imgsz, conf=conf, verbose=False)[0]
    return to_detections(result)


def track(tracker, frame, classes, imgsz=640):
    result = tracker.track(frame, classes=[class_ids[name] for name in classes], imgsz=imgsz, persist=True, verbose=False)[0]
    return [detection for detection in to_detections(result) if "id" in detection]

### Desenho e vídeo

As funções de desenho trabalham sobre imagens do OpenCV, com os canais na ordem BGR, então as cores são dadas como `(azul, verde, vermelho)`.

A `run_video` percorre um vídeo frame a frame. Cada frame é reduzido para a largura `width`, o que acelera tudo, e passa pelo rastreador. Em seguida, a função `process`, que muda de uma aplicação para outra, recebe o frame, as detecções rastreadas e o tempo `t` em segundos, aplica a lógica e desenha o resultado. A cada `show_every` frames, o frame desenhado é atualizado na saída da célula, e assim vemos o vídeo rodando. Passando `0` no lugar do caminho, a `run_video` lê da webcam.

In [ ]:
GREEN = (0, 200, 0)
RED = (0, 0, 230)
BLUE = (230, 120, 0)
YELLOW = (0, 220, 255)


def draw_box(image, box, color, text=None, thickness=2):
    x0, y0, x1, y1 = [int(v) for v in box]
    cv2.rectangle(image, (x0, y0), (x1, y1), color, thickness)
    if text is not None:
        cv2.putText(image, text, (x0, max(y0 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


def draw_panel(image, lines):
    # Caixa de texto no canto superior esquerdo
    height = 30 * len(lines) + 10
    width = max(len(line) for line in lines) * 14 + 20
    cv2.rectangle(image, (0, 0), (width, height), (0, 0, 0), -1)
    for k, line in enumerate(lines):
        cv2.putText(image, line, (10, 30 * (k + 1)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)


def show(image, width=10):
    plt.figure(figsize=(width, width * image.shape[0] / image.shape[1]))
    plt.imshow(image[:, :, ::-1])   # BGR -> RGB
    plt.axis("off")
    plt.show()


def run_video(source, classes, process, width=1280, imgsz=640, max_frames=None, show_every=5, display_width=800):
    tracker = YOLO("yolo26n.pt")   # rastreador novo a cada vídeo
    capture = cv2.VideoCapture(source)
    if not capture.isOpened():
        print(f"Não foi possível abrir {source}")
        return
    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    output = display(None, display_id=True)
    frame_index = 0
    start = time.time()
    while max_frames is None or frame_index < max_frames:
        ok, frame = capture.read()
        if not ok:
            break
        if source == 0:
            frame = cv2.flip(frame, 1)   # espelha a webcam
        h, w = frame.shape[:2]
        frame = cv2.resize(frame, (width, round(h * width / w)))

        tracks = track(tracker, frame, classes, imgsz)
        process(frame, tracks, frame_index / fps)

        if frame_index % show_every == 0:
            jpeg = cv2.imencode(".jpg", frame)[1].tobytes()
            output.update(IPImage(data=jpeg, width=display_width))
        frame_index += 1
    capture.release()
    print(f"{frame_index} frames em {time.time() - start:.0f} s")

## Monitor de estacionamento

Câmeras em estacionamentos de shoppings e aeroportos mostram, na entrada, quantas vagas estão livres em cada andar. A ideia é simples: a posição de cada vaga é marcada uma única vez, quando a câmera é instalada, e a cada nova imagem o sistema confere se há um carro sobre ela.

Vamos usar a câmera **UFPR04** do PKLot, que enxerga 28 vagas. As vagas foram anotadas no dataset e convertidas para caixas, e cada foto tem o gabarito de ocupação, com `1` para ocupada e `0` para livre, na mesma ordem das vagas.

In [ ]:
spots = [
    [644, 549, 805, 698], [701, 451, 849, 586], [747, 399, 879, 499], [771, 323, 900, 430],
    [810, 282, 918, 368], [832, 231, 938, 309], [851, 194, 956, 265], [874, 166, 978, 226],
    [897, 137, 982, 189], [470, 563, 648, 685], [678, 281, 797, 348], [711, 242, 827, 299],
    [736, 198, 853, 254], [760, 168, 868, 213], [798, 142, 885, 179], [361, 250, 438, 327],
    [408, 213, 486, 280], [458, 179, 534, 246], [495, 150, 569, 204], [530, 120, 606, 176],
    [563, 99, 635, 150], [189, 275, 272, 364], [252, 241, 334, 314], [316, 203, 391, 272],
    [366, 170, 435, 228], [410, 140, 480, 201], [450, 118, 522, 175], [491, 98, 554, 148],
]

parking_photos = {
    "ensolarado": ("data_4/2012-12-18_20_25_17.jpg", [1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1]),
    "nublado": ("data_7/2012-12-14_13_15_09.jpg", [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1]),
    "chuvoso": ("data_8/2012-12-21_18_20_14.jpg", [1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0]),
}

parking_images = {}
parking_truth = {}
for weather, (file, occupied) in parking_photos.items():
    path = download(PKLOT_URL + file, f"pklot_{weather}.jpg")
    parking_images[weather] = cv2.imread(path)
    parking_truth[weather] = occupied

image = parking_images["ensolarado"].copy()
for k, spot in enumerate(spots):
    draw_box(image, spot, YELLOW, str(k))
show(image)

Como as vagas são inclinadas na imagem, as caixas de vagas vizinhas se sobrepõem. Guarde isso: vai importar na hora de decidir se uma vaga está ocupada.

### Interseção sobre união

Para decidir se um carro está sobre uma vaga, comparamos as duas caixas com a **interseção sobre união** (IoU), a mesma medida que usamos para avaliar detectores:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

onde $|A|$ é a área da caixa $A$. A interseção de duas caixas alinhadas aos eixos também é uma caixa, com canto superior esquerdo $(\max(x_0^A, x_0^B), \max(y_0^A, y_0^B))$ e canto inferior direito $(\min(x_1^A, x_1^B), \min(y_1^A, y_1^B))$. Se as caixas não se tocam, essa "caixa" tem largura ou altura negativa, e a interseção é zero.

In [ ]:
def area(box):
    x0, y0, x1, y1 = box
    return max(0, x1 - x0) * max(0, y1 - y0)


def iou(box_a, box_b):
    intersection = area([
        max(box_a[0], box_b[0]), max(box_a[1], box_b[1]),
        min(box_a[2], box_b[2]), min(box_a[3], box_b[3]),
    ])
    union = area(box_a) + area(box_b) - intersection
    return intersection / union if union > 0 else 0

In [ ]:
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1
assert iou([0, 0, 10, 10], [5, 0, 15, 10]) == 50 / 150
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0
assert iou([0, 0, 10, 10], [10, 0, 20, 10]) == 0   # só encostam
print("iou ok")

### Vagas ocupadas

A `occupied_spots` recebe as caixas das vagas e as detecções de veículos e devolve, para cada vaga, `True` se ela está ocupada. Uma vaga está ocupada quando algum veículo tem IoU com ela acima de `threshold`.

In [ ]:
def occupied_spots(spots, detections, threshold):
    # TODO: devolva uma lista com um booleano por vaga, True se algum veículo
    # em detections (use detection["box"]) tem IoU com a vaga acima de threshold.
    # Dica: a função any(...) pode ajudar.
    raise NotImplementedError

In [ ]:
toy_spots = [[0, 0, 10, 20], [10, 0, 20, 20], [20, 0, 30, 20]]
toy_cars = [{"box": [1, 1, 10, 19]}, {"box": [21, 15, 29, 30]}]
assert occupied_spots(toy_spots, toy_cars, 0.5) == [True, False, False]
assert occupied_spots(toy_spots, toy_cars, 0.1) == [True, False, True]
assert occupied_spots(toy_spots, [], 0.5) == [False, False, False]
print("occupied_spots ok")

Agora juntamos tudo. A `check_parking` detecta os veículos de uma foto, aplica a `occupied_spots` e compara com o gabarito. Cada vaga é desenhada em verde se livre e em vermelho se ocupada; as caixas com borda grossa são os erros. As detecções aparecem em azul.

In [ ]:
vehicles = ["car", "truck", "bus", "motorcycle"]


def check_parking(weather, threshold=0.3, imgsz=640, draw=True):
    image = parking_images[weather].copy()
    detections = detect(image, vehicles, imgsz=imgsz)
    occupied = occupied_spots(spots, detections, threshold)
    truth = parking_truth[weather]
    accuracy = np.mean([o == t for o, t in zip(occupied, truth)])

    if draw:
        for detection in detections:
            draw_box(image, detection["box"], BLUE, thickness=1)
        for spot, o, t in zip(spots, occupied, truth):
            draw_box(image, spot, RED if o else GREEN, thickness=2 if o == t else 6)
        draw_panel(image, [f"livres: {occupied.count(False)}/{len(spots)}", f"acerto: {accuracy:.0%}"])
        show(image)
    return accuracy


check_parking("ensolarado")

O resultado não é bom: muitas vagas ocupadas aparecem como livres. Olhando as caixas azuis, o problema não está na nossa lógica, e sim na detecção: o YOLO não encontrou os carros pequenos, no fundo da imagem. Por padrão, o YOLO redimensiona a imagem para 640 pixels de largura antes de passá-la pela rede, e a foto tem 1280. Os carros distantes ficam com poucos pixels e somem. Vamos rodar na resolução original, com `imgsz=1280`.

In [ ]:
check_parking("ensolarado", imgsz=1280)

In [ ]:
for weather in parking_images:
    for imgsz in [640, 1280]:
        accuracy = check_parking(weather, imgsz=imgsz, draw=False)
        print(f"{weather:10s} imgsz {imgsz}: {accuracy:.0%}")

Com a resolução maior, o acerto passa de 85% nas três fotos. Vale olhar os erros que sobram na foto ensolarada: a vaga 9, no canto inferior esquerdo, tem um carro, mas está marcada como livre no gabarito. Anotações de datasets reais também erram, e um acerto de 100% nem sempre é possível, nem desejável.

Algumas observações sobre o que vimos:

- a configuração da detecção (resolução, tamanho do modelo, limiar de confiança) pesou muito mais no resultado do que a lógica de aplicação;
- o **limiar de IoU** é uma decisão de projeto. Como as caixas de vagas vizinhas se sobrepõem, um limiar baixo faz um carro ocupar duas vagas; um limiar alto faz um carro estacionado torto não ocupar nenhuma;
- as vagas foram marcadas para esta câmera. Se ela se mover, todas as caixas precisam ser refeitas. No PKLot, a UFPR04 tem três conjuntos de anotações, porque a câmera foi mexida durante a coleta.

## Contador de tráfego

Contagens de veículos alimentam o planejamento de semáforos, de obras e de transporte público. A forma clássica de contar é traçar uma **linha virtual** atravessando a via e contar cada veículo que a cruza. Como a câmera vê a rodovia de frente, uma linha horizontal $y = y_{\text{linha}}$ atravessa todas as faixas, e o sentido do cruzamento diz a mão de direção: quem desce na imagem vem em direção à câmera, quem sobe vai se afastando.

In [ ]:
line_y = 450

capture = cv2.VideoCapture(traffic_video)
ok, frame = capture.read()
capture.release()
frame = cv2.resize(frame, (1280, 720))
cv2.line(frame, (0, line_y), (1280, line_y), YELLOW, 3)
show(frame)

### Cruzando a linha

Um veículo cruzou a linha entre dois frames se o centro da sua caixa estava de um lado da linha no frame anterior e está do outro lado no frame atual. A `box_center`, já pronta, devolve o centro $(c_x, c_y)$ de uma caixa. A `crossed_line` recebe a caixa do frame anterior e a do atual e devolve:

- `"desce"`, se o centro estava acima da linha ($c_y < y_{\text{linha}}$) e agora está nela ou abaixo ($c_y \geq y_{\text{linha}}$);
- `"sobe"`, se estava nela ou abaixo e agora está acima;
- `None`, se não cruzou.

Lembre-se de que, em imagens, o eixo $y$ cresce para baixo.

In [ ]:
def box_center(box):
    x0, y0, x1, y1 = box
    return (x0 + x1) / 2, (y0 + y1) / 2


def crossed_line(previous_box, box, line_y):
    # TODO: devolva "desce", "sobe" ou None, comparando o y do centro
    # das duas caixas (use box_center) com line_y.
    raise NotImplementedError

In [ ]:
assert crossed_line([0, 80, 10, 90], [0, 95, 10, 105], 100) == "desce"
assert crossed_line([0, 95, 10, 105], [0, 80, 10, 90], 100) == "sobe"
assert crossed_line([0, 10, 10, 20], [0, 20, 10, 30], 100) is None
assert crossed_line([0, 95, 10, 105], [0, 100, 10, 110], 100) is None   # já tinha cruzado
print("crossed_line ok")

### Contagem

A `update_counts`, já pronta, é chamada a cada frame. Ela guarda em `last_box` a última caixa vista de cada `id` do rastreador e, quando a `crossed_line` indica um cruzamento, soma 1 em `counts[(classe, sentido)]`. Como cada veículo só passa de um lado para o outro uma vez, ele é contado uma única vez, desde que mantenha o mesmo `id`.

In [ ]:
def update_counts(counts, last_box, tracks, line_y):
    for detection in tracks:
        id = detection["id"]
        if id in last_box:
            direction = crossed_line(last_box[id], detection["box"], line_y)
            if direction is not None:
                key = (detection["label"], direction)
                counts[key] = counts.get(key, 0) + 1
        last_box[id] = detection["box"]

In [ ]:
counts, last_box = {}, {}
update_counts(counts, last_box, [{"id": 1, "label": "car", "box": [0, 80, 10, 90]}], 100)
update_counts(counts, last_box, [{"id": 1, "label": "car", "box": [0, 95, 10, 105]}, {"id": 2, "label": "bus", "box": [0, 95, 10, 105]}], 100)
update_counts(counts, last_box, [{"id": 1, "label": "car", "box": [0, 120, 10, 130]}, {"id": 2, "label": "bus", "box": [0, 80, 10, 90]}], 100)
assert counts == {("car", "desce"): 1, ("bus", "sobe"): 1}
print("update_counts ok")

In [ ]:
counts, last_box = {}, {}


def process_traffic(frame, tracks, t):
    update_counts(counts, last_box, tracks, line_y)

    cv2.line(frame, (0, line_y), (frame.shape[1], line_y), YELLOW, 3)
    for detection in tracks:
        draw_box(frame, detection["box"], GREEN, f"{detection['label']} {detection['id']}")
    lines = [f"{label} {direction}: {n}" for (label, direction), n in sorted(counts.items())]
    draw_panel(frame, lines or ["nenhum veículo ainda"])


run_video(traffic_video, vehicles, process_traffic)
print(counts)

Vale conferir a contagem assistindo ao vídeo. Erros típicos de contadores como este:

- um caminhão classificado como `car` em alguns frames e como `truck` em outros. A classe usada é a do frame do cruzamento, e um mesmo veículo pode ser contado em classes diferentes em vídeos diferentes;
- uma **troca de identidade** (*ID switch*) bem em cima da linha: quando o veículo é encoberto por alguns frames, o rastreador pode dar a ele um `id` novo, que não tem caixa anterior, e o cruzamento é perdido;
- com trânsito denso, veículos encobertos não são detectados e passam sem ser contados.